# VLM Architectures

**Module:** 15 — VLMs & Multimodal

Encoder, projector, and LLM families — training objectives and efficiency techniques.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Compare dual-encoder, fusion, adapter/projector, and native multimodal designs
- Explain contrastive vs generative objectives
- Map freeze/LoRA/distill/token-pruning efficiency levers
- Sketch a projector and a finetune plan


## Architectural Families

### Definition
Families differ in when/how vision and language interact: separate towers, deep fusion, shallow projectors, or joint transformers.

### Why it matters
Choice drives latency, cost, finetuneability, embeddings vs dialogue.

### How it works
Dual encoders for retrieval; projector+LLM for instruction VLMs; native MM for frontier joint training.

### Intuition
Retrieval wants dual towers; dialogue wants a decoder over visual tokens.

### Pitfalls
- CLIP alone for complex VQA
- Full finetune when LoRA suffices
- Ignoring tiling limits

### When to use
Model selection and finunetuning plans.


### Family comparison

| Family | Interaction | Use | Lineage |
|--------|-------------|-----|---------|
| Dual encoder | Late similarity | Retrieval | CLIP/SigLIP |
| Fusion | Deep cross-attn | Classic VQA | Flamingo-like |
| Projector+LLM | Visual tokens in LLM | Assistants | LLaVA-style |
| Native MM | Joint train | General MM | Frontier MM |

```mermaid
flowchart TB
  subgraph dual [Dual]
    I1[Img] --> S[Sim]
    T1[Txt] --> S
  end
  subgraph proj [Projector+LLM]
    I2[Img] --> P[Proj] --> L[LLM]
    T2[Txt] --> L
  end
```


In [ ]:
# Demo 1: toy dual-encoder retrieval
import math, random
random.seed(0)

def normalize(v):
    n = math.sqrt(sum(x*x for x in v)) or 1.0
    return [x/n for x in v]

def embed(text, dim=8):
    rnd = random.Random(hash(text) % (2**32))
    return normalize([rnd.uniform(-1,1) for _ in range(dim)])

def cosine(a,b): return sum(x*y for x,y in zip(a,b))
catalog = {k: embed(v) for k,v in {"img_cat":"photo of a cat","img_dog":"photo of a dog","img_invoice":"photo of an invoice"}.items()}
q = embed("find the invoice scan")
print(sorted(((cosine(q,v),k) for k,v in catalog.items()), reverse=True))


In [ ]:
# Demo 2: linear projector vision_dim -> llm_dim
import random
random.seed(1)

def matvec(W,x): return [sum(a*b for a,b in zip(row,x)) for row in W]
V,L,N = 16,32,4
W = [[random.uniform(-0.1,0.1) for _ in range(V)] for _ in range(L)]
vision = [[random.random() for _ in range(V)] for _ in range(N)]
proj = [matvec(W,t) for t in vision]
print("shape", len(proj), "x", len(proj[0]), "seq", 12+N)


## Training Objectives

### Definition
Contrastive loss aligns embeddings; generative loss teaches language about images; instruction tuning teaches useful answers; preference shapes style/safety.

### Why it matters
Objective mismatch explains why retrieval models fail at JSON extraction.

### How it works
Pretrain → SFT mixtures → optional preference opt; domain data for docs.

### Intuition
You get what you optimize.

### Pitfalls
- Expecting CLIP to emit JSON
- Tiny noisy SFT → regressions

### When to use
Choosing bases and planning finetunes.


### Objectives map

| Objective | Signal | Skill |
|-----------|--------|-------|
| Contrastive | Image↔caption match | Retrieval |
| Caption LM | Next-token | Description |
| Instruction | Supervised answers | Assistants |
| Grounding | Box/pointer | Localization |
| Preference | Rankings | Safer style |

**Efficiency:** freeze vision; LoRA LLM; distill; prune visual tokens; progressive resolution; quantize.


In [ ]:
# Demo 3: finetune plan
from dataclasses import dataclass

@dataclass
class FinetunePlan:
    freeze_vision: bool = True
    lora_llm: bool = True
    train_projector: bool = True
    full_llm: bool = False
    def parts(self):
        p = []
        if not self.freeze_vision: p.append("vision")
        if self.train_projector: p.append("projector")
        p.append("llm_full" if self.full_llm else "llm_lora" if self.lora_llm else "llm_frozen")
        return p
print(FinetunePlan().parts())
print(FinetunePlan(freeze_vision=False, full_llm=True, lora_llm=False).parts())


In [ ]:
# Demo 4: visual token pruning by energy
def prune(tokens, keep):
    scored = sorted(((sum(x*x for x in t), i, t) for i,t in enumerate(tokens)), reverse=True)
    return [t for _,_,t in sorted(scored[:keep], key=lambda z: z[1])]
toks = [[float(i), 0.1*i] for i in range(10)]
print([round(sum(x*x for x in t),2) for t in prune(toks,4)])


In [ ]:
# Demo 5: context cost estimate
def context_cost(tiles=4, tokens_per_tile=255, text_tokens=500, price_per_m=5.0):
    total = tiles*tokens_per_tile + text_tokens
    return {"tokens": total, "usd": total/1e6*price_per_m}
print(context_cost())
print(context_cost(tiles=1, tokens_per_tile=85, text_tokens=200, price_per_m=0.5))


In [ ]:
# Demo 6: InfoNCE-shaped toy loss on similarities
import math
def info_nce(sims, temperature=0.07):
    # sims[i][i] should be the positive
    losses = []
    for i, row in enumerate(sims):
        row_t = [s/temperature for s in row]
        m = max(row_t)
        exps = [math.exp(x-m) for x in row_t]
        losses.append(-math.log(exps[i]/sum(exps)))
    return sum(losses)/len(losses)
sims = [[0.9,0.1,0.2],[0.2,0.8,0.1],[0.1,0.2,0.85]]
print(round(info_nce(sims),4))


### Intuition

Dual encoders are two libraries with a shared catalog number system (embedding space).
Projector+LLM staples photo thumbnails into the novel the LLM is writing.


### Checklist — Architecture review

- [ ] Workload is retrieval vs generative (correct family)
- [ ] Resolution/tiling limits known
- [ ] Finetune plan lists frozen modules
- [ ] Serving latency fits SLA
- [ ] License/data terms reviewed


### Try it yourself — Architecture

1. Draw Flamingo-style cross-attn in mermaid.
2. Propose finetune plans for invoice JSON vs image search.
3. Estimate $ for 1M pages at 1200 tokens/page.

**Stretch:** Implement batched InfoNCE with more negatives.


### Try it yourself — Efficiency

1. Add random token drop as a baseline vs energy pruning.
2. List quantization tradeoffs for vision tower vs LLM.


## Knowledge Check

**Q1.** Why not use CLIP for multi-step UI agents?

<details><summary>Answer</summary>

It embeds/retrieves but lacks instruction-following generation and tool loops.

</details>

**Q2.** What does freezing the vision tower buy?

<details><summary>Answer</summary>

Lower train cost/stability; projector+LoRA often enough for domain adapt.

</details>


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `dual encoder` | Separate towers scored by similarity |
| `projector` | Vision→LLM dimension map |
| `LoRA` | Low-rank finetune adapters |
| `contrastive loss` | Pull matched pairs together |
| `visual tokens` | Projected patches in context |
| `native multimodal` | Jointly trained across modalities |


## Key Takeaways

- Match family to job: retrieval vs assistants
- Projector+LLM is the common instruction pattern
- Objectives determine emergent skills
- Freeze/LoRA/prune before full retrain


## Production Incident Patterns — VLM architecture

| Symptom | Likely cause | First fix |
|---------|--------------|-----------|
| Sudden cost spike | `detail=high` on huge pages | Resize + tile budget |
| Fluent wrong fields | VLM hallucination | OCR hybrid + schema |
| Cross-customer leak | Missing tenant filter | ACL in retriever code |
| Flaky eval scores | Unfrozen prompts/models | Pin versions + bakeoff set |
| Latency SLO burn | Full-page high detail | Crop ROI → mini model |

```
ASCII control loop:
  ingest -> normalize -> route model -> generate -> validate -> (HITL|export)
                     ^                              |
                     +-------- metrics/audit <------+
```


In [ ]:
# Cross-cutting: redact secrets before logging multimodal payloads
import re, json

SECRET_RE = re.compile(r"(api[_-]?key|bearer\s+[A-Za-z0-9._\-]+)", re.I)

def safe_log(payload: dict) -> str:
    s = json.dumps(payload)
    s = SECRET_RE.sub("***", s)
    if "base64," in s:
        s = re.sub(r"base64,[A-Za-z0-9+/=]+", "base64,[REDACTED]", s)
    return s[:500]

print(safe_log({
    "model": "gpt-4o",
    "api_key": "YOUR_OPENAI_API_KEY",
    "content": "data:image/png;base64,AAAABBBBCCCC",
    "topic": "VLM architecture",
}))


## Mini Case Study — VLM architecture

**Scenario:** A team ships a vision feature in one week. Demo looks great on three happy-path images.
**Week 2:** finance reports wrong totals; legal asks about image retention; GPU/API bill 4× forecast.

**Retro questions**
1. What was the output contract (schema) on day one?
2. Which failure mode had no metric?
3. Was there a crop/detail budget?
4. Who owns HITL and appeals?

**Design rule:** if a field can move money or identity, it needs a validator + disagreement path before automation.


In [ ]:
# Cross-cutting: simple SLO helper for vision endpoints
from dataclasses import dataclass

@dataclass
class VisionSLO:
    availability: float = 0.995
    p95_ms: int = 4000
    max_critical_field_error_rate: float = 0.005

def breached(slo: VisionSLO, avail: float, p95: int, crit_err: float) -> list[str]:
    out = []
    if avail < slo.availability: out.append("availability")
    if p95 > slo.p95_ms: out.append("latency")
    if crit_err > slo.max_critical_field_error_rate: out.append("critical_accuracy")
    return out or ["ok"]

print("VLM architecture", breached(VisionSLO(), 0.99, 5200, 0.02))


### Try it yourself — VLM architecture ops

1. Write a one-page runbook section for on-call when VLM architecture critical_accuracy SLO breaches.
2. Add a dashboard sketch: cost/1k images, CER/field error, HITL rate, p95 latency.
